In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/creditcardfraud/creditcard.csv


Install required libaries.

Import required libaries.

In [2]:
# import data science libraries
import pandas as pd
import numpy as np
import math


# import pytorch libraries
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# import utility libraries
from tqdm import tqdm
from datetime import datetime

# import visualisation libraries
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score
from sklearn.preprocessing import QuantileTransformer


In [3]:
import os
import pandas as pd

if os.path.exists("/kaggle/input/creditcardfraud/creditcard.csv"):
    data_path = "/kaggle/input/creditcardfraud/creditcard.csv"
else:
    data_path = "./data/creditcard.csv"

df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
# from sklearn.model_selection import train_test_split

# # 전체 데이터에서 5%만 사용 (테스트용)
# df_small, _ = train_test_split(
#     df,
#     test_size=0.95,              # 🔥 5%만 남김
#     stratify=df["Class"],        # Fraud 비율 유지
#     random_state=42
# )

# print("Small DF shape:", df_small.shape)
# print("Fraud ratio:", df_small["Class"].mean())


In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["Class"], random_state=42
)
# train_df, test_df = train_test_split(
#     df_small, test_size=0.2, stratify=df_small["Class"], random_state=42
# )
train_base, val_df = train_test_split(
    train_df, test_size=0.2, stratify=train_df["Class"], random_state=42
)

FEATURE_COLS = [c for c in df.columns if c != "Class"]

In [6]:
def print_class_distribution(name, df):
    counts = df["Class"].value_counts().sort_index()
    total = len(df)

    n_normal = counts.get(0, 0)
    n_fraud  = counts.get(1, 0)

    r_normal = n_normal / total
    r_fraud  = n_fraud / total

    print(f"\n[{name}]")
    print(f"Total: {total}")
    print(f"Normal (0): {n_normal}  ({r_normal:.6f})")
    print(f"Fraud  (1): {n_fraud}  ({r_fraud:.6f})")


In [7]:
print_class_distribution("FULL df", df)
print_class_distribution("train_df", train_df)
print_class_distribution("train_base", train_base)
print_class_distribution("val_df", val_df)
print_class_distribution("test_df", test_df)



[FULL df]
Total: 284807
Normal (0): 284315  (0.998273)
Fraud  (1): 492  (0.001727)

[train_df]
Total: 227845
Normal (0): 227451  (0.998271)
Fraud  (1): 394  (0.001729)

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[val_df]
Total: 45569
Normal (0): 45490  (0.998266)
Fraud  (1): 79  (0.001734)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


In [8]:
# ===== XGBoost imbalance weight =====
# scale_pos_weight = (
#     train_base["Class"].value_counts()[0] /
#     train_base["Class"].value_counts()[1]
# )
scale_pos_weight = (
    train_df["Class"].value_counts()[0] /
    train_df["Class"].value_counts()[1]
)

print("scale_pos_weight:", scale_pos_weight)

XGB_PARAMS_BASE = dict(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

XGB_PARAMS_ON  = {**XGB_PARAMS_BASE, "scale_pos_weight": scale_pos_weight}
XGB_PARAMS_OFF = {**XGB_PARAMS_BASE}

def make_hybrid(real_df, syn_df):
    return pd.concat([real_df, syn_df], axis=0)\
             .sample(frac=1, random_state=42)\
             .reset_index(drop=True)


scale_pos_weight: 577.2868020304569


In [9]:
def find_best_threshold(prob, y_true):
    thresholds = np.linspace(0.001, 0.5, 200)
    best_t, best_f1 = 0.5, 0
    for t in thresholds:
        pred = (prob > t).astype(int)
        f1 = f1_score(y_true, pred)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


Init and set experiment parameters.

In [10]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

FINDIFF_BASE = dict(
    seed=1234,
    activation="lrelu",
    diffusion_steps=500,
    beta_start=1e-4,
    beta_end=0.02,
    scheduler="linear",
    epochs=500,
    batch_size=512,
    lr=1e-4,
    mlp_layers=[1024,1024,1024,1024],
    scaler="standard",   # "standard" or "quantile"
    device=device
)


Device: cuda


Set random seed values.

In [11]:
seed = 1234

# set numpy seed
np.random.seed(seed)

# set pytorch seed
torch.manual_seed(seed)

# set cuda seed
torch.cuda.manual_seed(seed)

# **Load, pre-process, and init the UCU Credit Card dataset**

Set numerical and categorical dataset attributes.

Transform the numerical attributes.

In [12]:
from sklearn.preprocessing import StandardScaler, QuantileTransformer

def build_diff_scaler(name, seed=1234):
    if name == "standard":
        return StandardScaler()
    elif name == "quantile":
        return QuantileTransformer(
            n_quantiles=1000,
            output_distribution="normal",
            random_state=seed
        )
    else:
        raise ValueError("scaler must be 'standard' or 'quantile'")


Transform the categorical attributes.

Convert numerical and categorical attributes as well as the labels to tensors.

Convert dataset to tensor dataset.

Init the data loader.

# **Implement the FinDiff model**

Implement the FinDiff backbone model.

In [13]:
# define base feedforward network
class BaseNetwork(nn.Module):

    # define base network constructor
    def __init__(self, hidden_size, activation='lrelu'):

        # call super calass constructor 
        super(BaseNetwork, self).__init__()

        # init 
        self.layers = self.init_layers(hidden_size)

        # case: lrelu activation
        if activation == 'lrelu':

            # set lrelu activation
            self.activation = nn.LeakyReLU(negative_slope=0.4, inplace=True)

        # case: relu activation
        elif activation == 'relu':

            # set relu activation
            self.activation = nn.ReLU(inplace=True)

        # case: tanh activation
        elif activation == 'tanh':

            # set tanh activation
            self.activation = nn.Tanh()

        # case: sigmoid activation
        else:

            # set sigmoid activation
            self.activation = nn.Sigmoid()

    # define layer initialization 
    def init_layers(self, layer_dimensions):

        # init layers
        layers = []

        # iterate over layer dimensions 
        for i in range(len(layer_dimensions)-1):

            # init linear layer 
            layer = nn.Linear(layer_dimensions[i], layer_dimensions[i + 1], bias=True)
            
            # init linear layer weights
            nn.init.xavier_uniform_(layer.weight)
            
            # init linear layer bias
            nn.init.constant_(layer.bias, 0.0)

            # collecet linear layer 
            layers.append(layer)
            
            # register linear layer parameters
            self.add_module('linear_' + str(i), layer)

        # return layers
        return layers

    # define forward pass
    def forward(self, x):

        # iterate over layers
        for i in range(len(self.layers)):

            # run layer forward pass 
            x = self.activation(self.layers[i](x))

        # return forward pass result
        return x

Implement the FinDiff model synthesizer.

In [14]:

# define MLP synthesizer network
class MLPSynthesizer(nn.Module):
    def __init__(self, d_in, hidden_layers, dim_t=64, activation='lrelu', n_classes=2):
        super().__init__()

        self.dim_t = dim_t

        self.backbone = BaseNetwork([dim_t, *hidden_layers], activation=activation)

        self.label_embedding = nn.Embedding(n_classes, dim_t)

        self.projection = nn.Sequential(
            nn.Linear(d_in, dim_t),
            nn.SiLU(),
            nn.Linear(dim_t, dim_t)
        )

        self.time_embed = nn.Sequential(
            nn.Linear(dim_t, dim_t),
            nn.SiLU(),
            nn.Linear(dim_t, dim_t)
        )

        self.head = nn.Linear(hidden_layers[-1], d_in)

    def embed_time(self, timesteps, dim_out, max_period=10000):
        half = dim_out // 2
        freqs = torch.exp(-math.log(max_period) * torch.arange(half) / half).to(timesteps.device)
        args = timesteps[:, None].float() * freqs[None]
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        if dim_out % 2:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
        return emb

    def forward(self, x, timesteps, label):
        time_emb = self.embed_time(timesteps, self.dim_t)
        time_emb = self.time_embed(time_emb)

        time_label_emb = time_emb + self.label_embedding(label)

        x = self.projection(x)
        x = x + time_label_emb
        x = self.backbone(x)
        return self.head(x)


Implement the FinDiff model base diffuser.

In [15]:
# define BaseDiffuser network
class BaseDiffuser(object):

    # define base diffuser network constructor
    def __init__(
            self, 
            total_steps=1000, 
            beta_start=1e-4, 
            beta_end=0.02, 
            device='cpu',
            scheduler='linear'
        ):

        # set diffusion steps
        self.total_steps = total_steps

        # set diffusion start beta
        self.beta_start = beta_start

        # set diffusion end beta
        self.beta_end = beta_end

        # set compute device
        self.device = device

        # set noise schedule alphas and betas
        self.alphas, self.betas = self.prepare_noise_schedule(scheduler=scheduler)

        # set noise schedule alhpa hats
        self.alphas_hat = torch.cumprod(self.alphas, dim=0)

    # define noise schedule
    def prepare_noise_schedule(self, scheduler: str):

        # determine noise scheduler scale
        scale = 1000 / self.total_steps

        # scale beta start
        beta_start = scale * self.beta_start

        # scale beta end
        beta_end = scale * self.beta_end

        # case: linear noise scheduler
        if scheduler == 'linear':

            # determine linear noise schedule betas
            betas = torch.linspace(beta_start, beta_end, self.total_steps)

            # determine linear noise schedule alphas
            alphas = 1.0 - betas

        # case: quadratic noise scheduler
        elif scheduler == 'quad':

            # determine quadratic noise schedule betas
            betas = torch.linspace(self.beta_start ** 0.5, self.beta_end ** 0.5, self.total_steps) ** 2

            # determine quadratic noise schedule alphas 
            alphas = 1.0 - betas

        # return noise scheduler alphas and betas
        return alphas.to(self.device), betas.to(self.device)

    # define random timesteps sampler 
    def sample_random_timesteps(self, n: int):

        # sample random timesteps
        t = torch.randint(low=1, high=self.total_steps, size=(n,), device=self.device)

        # return random timesteps
        return t

    # define gaussian noise addition
    def add_gauss_noise(self, x_num, t):

        # determine noise alpha hat
        sqrt_alpha_hat = torch.sqrt(self.alphas_hat[t])[:, None]

        # determine noise one minius alpha hat 
        sqrt_one_minus_alpha_hat = torch.sqrt(1 - self.alphas_hat[t])[:, None]

        # determine numeric noise
        noise_num = torch.randn_like(x_num)

        # determine x numeric noise
        x_noise_num = sqrt_alpha_hat * x_num + sqrt_one_minus_alpha_hat * noise_num

        # return x numeric noise and numeric noise
        return x_noise_num, noise_num

    # define gaussian noise sampling
    def p_sample_gauss(self, model_out, z_norm, timesteps):

        # determine noise alpha hat
        sqrt_alpha_t = torch.sqrt(self.alphas[timesteps])[:, None]

        # determine noise betas
        betas_t = self.betas[timesteps][:, None]
        
        # determine noise one minius alpha hat 
        sqrt_one_minus_alpha_hat_t = torch.sqrt(1 - self.alphas_hat[timesteps])[:, None]
        
        epsilon_t = torch.sqrt(self.betas[timesteps][:, None])

        # determine random noise
        random_noise = torch.randn_like(z_norm)
        random_noise[timesteps == 0] = 0.0

        # determine model mean
        model_mean = ((1 / sqrt_alpha_t) * (z_norm - (betas_t * model_out / sqrt_one_minus_alpha_hat_t)))

        # determine z norm
        z_norm = model_mean + (epsilon_t * random_noise)

        # return z norm
        return z_norm

# **Initialize and train the FinDiff model**

In [16]:
from sklearn.metrics import pairwise_distances
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Distribution / Collapse
from scipy.stats import ks_2samp, wasserstein_distance

def distribution_similarity(real_df, syn_df, exclude_cols=["Class"]):
    ks_scores, wass_scores = [], []
    cols = [c for c in real_df.columns if c not in exclude_cols]
    for col in cols:
        ks_scores.append(ks_2samp(real_df[col].values, syn_df[col].values).statistic)
        wass_scores.append(wasserstein_distance(real_df[col].values, syn_df[col].values))
    return {
        "KS_mean": float(np.mean(ks_scores)),
        "Wasserstein_mean": float(np.mean(wass_scores))
    }

def fraud_collapse_check(syn_df, threshold=0.001):
    fr = float(syn_df["Class"].mean())
    return {"Fraud_Ratio": fr, "Collapsed": fr < threshold}

# Privacy
def privacy_nn_memorization(real_df, syn_df, feature_cols, threshold=1e-6,
                            sample_real=5000, sample_syn=5000, seed=42):
    real_s = real_df.sample(n=min(sample_real, len(real_df)), random_state=seed)
    syn_s  = syn_df.sample(n=min(sample_syn, len(syn_df)), random_state=seed)

    real_X = real_s[feature_cols].values
    syn_X  = syn_s[feature_cols].values

    dists = pairwise_distances(syn_X, real_X, metric="euclidean")
    min_dists = dists.min(axis=1)

    out = {
        "NN_mean": float(min_dists.mean()),
        "NN_min": float(min_dists.min()),
        "NearDup_rate(thr)": float((min_dists < threshold).mean())
    }

    real_fraud = real_df[real_df.Class==1]
    syn_fraud  = syn_df[syn_df.Class==1]
    if len(real_fraud) > 0 and len(syn_fraud) > 0:
        rf = real_fraud.sample(n=min(2000, len(real_fraud)), random_state=seed)[feature_cols].values
        sf = syn_fraud.sample(n=min(2000, len(syn_fraud)), random_state=seed)[feature_cols].values
        d_f = pairwise_distances(sf, rf, metric="euclidean")
        min_d_f = d_f.min(axis=1)
        out["Fraud_NN_mean"] = float(min_d_f.mean())
        out["Fraud_NN_min"]  = float(min_d_f.min())
    else:
        out["Fraud_NN_mean"] = np.nan
        out["Fraud_NN_min"]  = np.nan

    return out

# XGB eval
def evaluate_holdout_xgb_simple(train_any_df, test_df, xgb_params, threshold=0.5):
    X_train = train_any_df.drop(columns=["Class"])
    y_train = train_any_df["Class"]

    X_test  = test_df.drop(columns=["Class"])
    y_test  = test_df["Class"]

    def _ratio(y, name):
        counts = y.value_counts().sort_index()
        total = len(y)
        n0 = counts.get(0, 0)
        n1 = counts.get(1, 0)
        print(f"[{name}] Normal: {n0/total:.6f}  Fraud: {n1/total:.6f}")

    _ratio(y_train, "Train ratio")
    _ratio(y_test,  "Test ratio")

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    model = XGBClassifier(**xgb_params)
    model.fit(X_train_s, y_train)

    prob = model.predict_proba(X_test_s)[:, 1]
    pred = (prob > threshold).astype(int)

    return {
        "ROC-AUC": roc_auc_score(y_test, prob),
        "AUPRC": average_precision_score(y_test, prob),
        "F1": f1_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
    }

# Natural sampling only
def sample_diffusion_natural(model, diffuser, diff_scaler, n, d_in, p_fraud, feature_cols, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    labels = (np.random.rand(n) < p_fraud).astype(int)
    labels_t = torch.LongTensor(labels).to(device)

    x = torch.randn((n, d_in), device=device)

    with torch.no_grad():
        for step in reversed(range(diffuser.total_steps)):
            t = torch.full((n,), step, dtype=torch.long, device=device)
            eps = model(x, t, labels_t)
            x = diffuser.p_sample_gauss(eps, x, t)

    x = diff_scaler.inverse_transform(x.cpu().numpy())
    syn_df = pd.DataFrame(x, columns=feature_cols)
    syn_df["Class"] = labels
    return syn_df


In [17]:
def run_findiff_sweep(
    param_name, param_values,
    base_cfg,
    train_base, test_df,
    feature_cols,
    XGB_PARAMS_ON, XGB_PARAMS_OFF,
    do_privacy=True, privacy_only_values=None,
    save_prefix="findiff"
):
    results = {}
    quality = {}
    privacy_rows = []

    p_fraud = float(train_base["Class"].mean())
    d_in = len(feature_cols)

    for v in param_values:
        setting = f"{param_name}={v}"
        print(f"\n[FiNDiff sweep] {setting}")

        print_class_distribution("train_base", train_base)
        print_class_distribution("test_df", test_df)

        cfg = dict(base_cfg)
        cfg[param_name] = v

        # seeds
        np.random.seed(cfg["seed"])
        torch.manual_seed(cfg["seed"])
        torch.cuda.manual_seed(cfg["seed"])

        # scaler + data
        diff_scaler = build_diff_scaler(cfg["scaler"], seed=cfg["seed"])
        X_train_np = diff_scaler.fit_transform(train_base[feature_cols])
        train_x = torch.FloatTensor(X_train_np)
        train_y = torch.LongTensor(train_base["Class"].values.astype(int))

        train_set = TensorDataset(train_x, train_y)
        dataloader = DataLoader(train_set, batch_size=cfg["batch_size"], shuffle=True)

        # build models
        synthesizer = MLPSynthesizer(
            d_in=d_in,
            hidden_layers=cfg["mlp_layers"],
            activation=cfg["activation"],
            n_classes=2
        ).to(cfg["device"])

        diffuser = BaseDiffuser(
            total_steps=cfg["diffusion_steps"],
            beta_start=cfg["beta_start"],
            beta_end=cfg["beta_end"],
            scheduler=cfg["scheduler"],
            device=cfg["device"]
        )

        optimizer = optim.Adam(synthesizer.parameters(), lr=cfg["lr"])
        lr_scheduler = CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
        loss_fnc = nn.MSELoss()

        # train
        synthesizer.train()
        epoch_losses = []

        for epoch in tqdm(range(cfg["epochs"]), desc=f"train {setting}"):
            batch_losses = []
            for batch_x, batch_y in dataloader:
                batch_x = batch_x.to(cfg["device"])
                batch_y = batch_y.to(cfg["device"])

                t = diffuser.sample_random_timesteps(batch_x.size(0))
                x_t, noise = diffuser.add_gauss_noise(batch_x, t)
                pred_noise = synthesizer(x_t, t, batch_y)

                loss = loss_fnc(pred_noise, noise)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                batch_losses.append(loss.item())

            lr_scheduler.step()
            epoch_losses.append(float(np.mean(batch_losses)))

        # sample natural
        syn = sample_diffusion_natural(
            synthesizer, diffuser, diff_scaler,
            n=len(train_base),
            d_in=d_in,
            p_fraud=p_fraud,
            feature_cols=feature_cols,
            device=cfg["device"]
        )

        # performance
        uniq = np.unique(syn["Class"].values)
        if len(uniq) >= 2:
            results[(setting, "syn_only_spw_ON")]  = evaluate_holdout_xgb_simple(syn, test_df, XGB_PARAMS_ON)
            results[(setting, "syn_only_spw_OFF")] = evaluate_holdout_xgb_simple(syn, test_df, XGB_PARAMS_OFF)
        else:
            print(f"  -> skip syn-only (only one class in syn): {uniq}")

        hybrid = make_hybrid(train_base, syn)
        results[(setting, "hybrid_spw_ON")]  = evaluate_holdout_xgb_simple(hybrid, test_df, XGB_PARAMS_ON)
        results[(setting, "hybrid_spw_OFF")] = evaluate_holdout_xgb_simple(hybrid, test_df, XGB_PARAMS_OFF)

        # privacy
        if do_privacy:
            run_priv = True
            if privacy_only_values is not None:
                run_priv = (v in privacy_only_values)
            if run_priv:
                priv = privacy_nn_memorization(
                    real_df=train_base,
                    syn_df=syn,
                    feature_cols=feature_cols,
                    threshold=1e-6,
                    sample_real=5000,
                    sample_syn=5000,
                    seed=42
                )
                privacy_rows.append({"setting": setting, **priv})

        # quality
        real_sample = train_base.sample(n=min(len(train_base), len(syn)), replace=False, random_state=42)
        syn_sample  = syn.sample(n=len(real_sample), random_state=42)

        quality[setting] = {
            **distribution_similarity(real_sample, syn_sample),
            **fraud_collapse_check(syn),
            "Diffusion_Loss_Var(last100)": float(np.var(epoch_losses[-100:])) if len(epoch_losses) >= 100 else float(np.var(epoch_losses))
        }

    # to DataFrames 
    perf_df = pd.DataFrame(results).T
    perf_df.index = pd.MultiIndex.from_tuples(perf_df.index, names=[param_name, "train_setting"])
    perf_df = perf_df.sort_index()

    quality_df = pd.DataFrame(quality).T.sort_index()
    privacy_df = pd.DataFrame(privacy_rows).set_index("setting") if len(privacy_rows) else pd.DataFrame()

    # save CSV
    perf_path   = f"{save_prefix}_{param_name}_perf.csv"
    qual_path   = f"{save_prefix}_{param_name}_quality.csv"
    priv_path   = f"{save_prefix}_{param_name}_privacy.csv"

    perf_df.to_csv(perf_path)
    quality_df.to_csv(qual_path)
    privacy_df.to_csv(priv_path)

    # print tables
    display(perf_df)
    display(quality_df)
    display(privacy_df)

    print("Saved:", perf_path, qual_path, priv_path)

    return perf_df, quality_df, privacy_df

In [18]:
epochs_list = [100, 300]
perf_ep, qual_ep, priv_ep = run_findiff_sweep(
    param_name="epochs",
    param_values=epochs_list,
    base_cfg=FINDIFF_BASE,
    train_base=train_base,
    test_df=test_df,
    feature_cols=FEATURE_COLS,
    XGB_PARAMS_ON=XGB_PARAMS_ON,
    XGB_PARAMS_OFF=XGB_PARAMS_OFF,
    do_privacy=True,
    privacy_only_values=[100, 300],
    save_prefix="findiff"
)


[FiNDiff sweep] epochs=100

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train epochs=100: 100%|██████████| 100/100 [04:06<00:00,  2.46s/it]


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720

[FiNDiff sweep] epochs=300

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train epochs=300: 100%|██████████| 300/300 [12:28<00:00,  2.50s/it]


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720


ROC-AUC     AUPRC        F1    Recall
epochs     train_setting                                           
epochs=100 hybrid_spw_OFF    0.976242  0.848188  0.843243  0.795918
           hybrid_spw_ON     0.974145  0.873015  0.857143  0.857143
           syn_only_spw_OFF  0.960969  0.659836  0.797980  0.806122
           syn_only_spw_ON   0.973171  0.617813  0.759259  0.836735
epochs=300 hybrid_spw_OFF    0.969477  0.859708  0.867725  0.836735
           hybrid_spw_ON     0.973447  0.859292  0.842640  0.846939
           syn_only_spw_OFF  0.962538  0.759027  0.790000  0.806122
           syn_only_spw_ON   0.965210  0.765642  0.794118  0.826531

,KS_mean,Wasserstein_mean,Fraud_Ratio,Collapsed,Diffusion_Loss_Var(last100)
epochs=100,0.033372,169.775039,0.001799,False,0.002255
epochs=300,0.019282,86.657589,0.001799,False,0.000001


,NN_mean,NN_min,NearDup_rate(thr),Fraud_NN_mean,Fraud_NN_min
setting,,,,,
epochs=100,670.974892,2.847864,0.0,116525.408142,58.409029
epochs=300,382.716492,2.482519,0.0,105427.636821,34.004625


Saved: findiff_epochs_perf.csv findiff_epochs_quality.csv findiff_epochs_privacy.csv


In [19]:
mlp_list = [
    [256,256],
    [512,512,512]
]

perf_mlp, qual_mlp, priv_mlp = run_findiff_sweep(
    param_name="mlp_layers",
    param_values=mlp_list,
    base_cfg=FINDIFF_BASE,
    train_base=train_base,
    test_df=test_df,
    feature_cols=FEATURE_COLS,
    XGB_PARAMS_ON=XGB_PARAMS_ON,
    XGB_PARAMS_OFF=XGB_PARAMS_OFF,
    do_privacy=True,
    privacy_only_values=mlp_list,
    save_prefix="findiff"
)


[FiNDiff sweep] mlp_layers=[256, 256]

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train mlp_layers=[256, 256]: 100%|██████████| 500/500 [17:55<00:00,  2.15s/it]


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720

[FiNDiff sweep] mlp_layers=[512, 512, 512]

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train mlp_layers=[512, 512, 512]: 100%|██████████| 500/500 [18:28<00:00,  2.22s/it]


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720


ROC-AUC     AUPRC        F1  \
mlp_layers                 train_setting                                    
mlp_layers=[256, 256]      hybrid_spw_OFF    0.977629  0.855584  0.857143   
                           hybrid_spw_ON     0.986827  0.867361  0.850000   
                           syn_only_spw_OFF  0.971867  0.699848  0.792079   
                           syn_only_spw_ON   0.969641  0.690916  0.782609   
mlp_layers=[512, 512, 512] hybrid_spw_OFF    0.977697  0.877227  0.869110   
                           hybrid_spw_ON     0.977131  0.875965  0.815534   
                           syn_only_spw_OFF  0.961620  0.659476  0.780952   
                           syn_only_spw_ON   0.965877  0.652436  0.728889   

                                               Recall  
mlp_layers                 train_setting               
mlp_layers=[256, 256]      hybrid_spw_OFF    0.826531  
                           hybrid_spw_ON     0.867347  
                           syn_only_spw_OFF  0.816327  
                           syn_only_spw_ON   0.826531  
mlp_layers=[512, 512, 512] hybrid_spw_OFF    0.846939  
                           hybrid_spw_ON     0.857143  
                           syn_only_spw_OFF  0.836735  
                           syn_only_spw_ON   0.836735

,KS_mean,Wasserstein_mean,Fraud_Ratio,Collapsed,Diffusion_Loss_Var(last100)
"mlp_layers=[256, 256]",0.04195,380.188335,0.001799,False,0.0
"mlp_layers=[512, 512, 512]",0.029023,172.968763,0.001799,False,0.0


,NN_mean,NN_min,NearDup_rate(thr),Fraud_NN_mean,Fraud_NN_min
setting,,,,,
"mlp_layers=[256, 256]",2569.341217,3.340480,0.0,24336.205742,18.051167
"mlp_layers=[512, 512, 512]",1810.557559,2.701083,0.0,41004.183066,17.512423


Saved: findiff_mlp_layers_perf.csv findiff_mlp_layers_quality.csv findiff_mlp_layers_privacy.csv


In [20]:
steps_list = [200, 800]
perf_steps, qual_steps, priv_steps = run_findiff_sweep(
    param_name="diffusion_steps",
    param_values=steps_list,
    base_cfg=FINDIFF_BASE,
    train_base=train_base,
    test_df=test_df,
    feature_cols=FEATURE_COLS,
    XGB_PARAMS_ON=XGB_PARAMS_ON,
    XGB_PARAMS_OFF=XGB_PARAMS_OFF,
    do_privacy=True,
    privacy_only_values=[200, 800],
    save_prefix="findiff"
)


[FiNDiff sweep] diffusion_steps=200

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train diffusion_steps=200: 100%|██████████| 500/500 [20:35<00:00,  2.47s/it]


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720

[FiNDiff sweep] diffusion_steps=800

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train diffusion_steps=800: 100%|██████████| 500/500 [20:50<00:00,  2.50s/it]


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720


ROC-AUC     AUPRC        F1    Recall
diffusion_steps     train_setting                                           
diffusion_steps=200 hybrid_spw_OFF    0.983880  0.855560  0.867725  0.836735
                    hybrid_spw_ON     0.982376  0.862127  0.841584  0.867347
                    syn_only_spw_OFF  0.979626  0.736920  0.796020  0.816327
                    syn_only_spw_ON   0.984126  0.761623  0.777251  0.836735
diffusion_steps=800 hybrid_spw_OFF    0.975751  0.856618  0.851064  0.816327
                    hybrid_spw_ON     0.979096  0.870878  0.807692  0.857143
                    syn_only_spw_OFF  0.969343  0.704862  0.786408  0.826531
                    syn_only_spw_ON   0.975002  0.710125  0.790476  0.846939

,KS_mean,Wasserstein_mean,Fraud_Ratio,Collapsed,Diffusion_Loss_Var(last100)
diffusion_steps=200,0.016774,68.935399,0.001799,False,0.0
diffusion_steps=800,0.01622,69.689237,0.001799,False,0.0


,NN_mean,NN_min,NearDup_rate(thr),Fraud_NN_mean,Fraud_NN_min
setting,,,,,
diffusion_steps=200,351.283825,3.270137,0.0,39232.599189,19.117494
diffusion_steps=800,251.969155,2.256389,0.0,69975.391917,14.802430


Saved: findiff_diffusion_steps_perf.csv findiff_diffusion_steps_quality.csv findiff_diffusion_steps_privacy.csv


In [21]:
scalers = ["standard", "quantile"]
perf_sc, qual_sc, priv_sc = run_findiff_sweep(
    param_name="scaler",
    param_values=scalers,
    base_cfg=FINDIFF_BASE,
    train_base=train_base,
    test_df=test_df,
    feature_cols=FEATURE_COLS,
    XGB_PARAMS_ON=XGB_PARAMS_ON,
    XGB_PARAMS_OFF=XGB_PARAMS_OFF,
    do_privacy=True,
    privacy_only_values=["standard"],
    save_prefix="findiff"
)




[FiNDiff sweep] scaler=standard

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train scaler=standard: 100%|██████████| 500/500 [20:48<00:00,  2.50s/it]


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720

[FiNDiff sweep] scaler=quantile

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train scaler=quantile: 100%|██████████| 500/500 [20:47<00:00,  2.49s/it]
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720


ROC-AUC     AUPRC        F1    Recall
scaler          train_setting                                           
scaler=quantile hybrid_spw_OFF    0.978765  0.854867  0.851064  0.816327
                hybrid_spw_ON     0.983517  0.858186  0.844221  0.857143
                syn_only_spw_OFF  0.969653  0.749906  0.796020  0.816327
                syn_only_spw_ON   0.971289  0.693022  0.798030  0.826531
scaler=standard hybrid_spw_OFF    0.978017  0.872985  0.873684  0.846939
                hybrid_spw_ON     0.979233  0.873741  0.823529  0.857143
                syn_only_spw_OFF  0.963281  0.724658  0.790244  0.826531
                syn_only_spw_ON   0.973218  0.745546  0.762791  0.836735

,KS_mean,Wasserstein_mean,Fraud_Ratio,Collapsed,Diffusion_Loss_Var(last100)
scaler=quantile,0.017269,45.31178,0.001799,False,0.0
scaler=standard,0.017075,70.640836,0.001799,False,0.0


,NN_mean,NN_min,NearDup_rate(thr),Fraud_NN_mean,Fraud_NN_min
setting,,,,,
scaler=standard,222.094893,2.329104,0.0,55659.515554,14.185173


Saved: findiff_scaler_perf.csv findiff_scaler_quality.csv findiff_scaler_privacy.csv


In [22]:
batch_list = [256, 768]
perf_bs, qual_bs, priv_bs = run_findiff_sweep(
    param_name="batch_size",
    param_values=batch_list,
    base_cfg=FINDIFF_BASE,
    train_base=train_base,
    test_df=test_df,
    feature_cols=FEATURE_COLS,
    XGB_PARAMS_ON=XGB_PARAMS_ON,
    XGB_PARAMS_OFF=XGB_PARAMS_OFF,
    do_privacy=True,
    privacy_only_values=[256, 768],
    save_prefix="findiff"
)


[FiNDiff sweep] batch_size=256

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train batch_size=256: 100%|██████████| 500/500 [28:51<00:00,  3.46s/it]


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720

[FiNDiff sweep] batch_size=768

[train_base]
Total: 182276
Normal (0): 181961  (0.998272)
Fraud  (1): 315  (0.001728)

[test_df]
Total: 56962
Normal (0): 56864  (0.998280)
Fraud  (1): 98  (0.001720)


train batch_size=768: 100%|██████████| 500/500 [19:10<00:00,  2.30s/it]


[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998201  Fraud: 0.001799
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720
[Train ratio] Normal: 0.998236  Fraud: 0.001764
[Test ratio] Normal: 0.998280  Fraud: 0.001720


ROC-AUC     AUPRC        F1    Recall
batch_size     train_setting                                           
batch_size=256 hybrid_spw_OFF    0.978569  0.859032  0.864865  0.816327
               hybrid_spw_ON     0.976146  0.871160  0.815920  0.836735
               syn_only_spw_OFF  0.967146  0.760434  0.790244  0.826531
               syn_only_spw_ON   0.967088  0.744575  0.760563  0.826531
batch_size=768 hybrid_spw_OFF    0.975680  0.852219  0.878307  0.846939
               hybrid_spw_ON     0.968928  0.863947  0.844221  0.857143
               syn_only_spw_OFF  0.967223  0.757432  0.796117  0.836735
               syn_only_spw_ON   0.962649  0.718410  0.788462  0.836735

,KS_mean,Wasserstein_mean,Fraud_Ratio,Collapsed,Diffusion_Loss_Var(last100)
batch_size=256,0.01388,61.366171,0.001799,False,0.0
batch_size=768,0.018179,77.20752,0.001799,False,0.0


,NN_mean,NN_min,NearDup_rate(thr),Fraud_NN_mean,Fraud_NN_min
setting,,,,,
batch_size=256,259.357353,2.962298,0.0,28020.274238,12.616563
batch_size=768,464.080192,1.318377,0.0,59451.608754,19.186841


Saved: findiff_batch_size_perf.csv findiff_batch_size_quality.csv findiff_batch_size_privacy.csv


In [23]:
# from sklearn.metrics import pairwise_distances
# import numpy as np

# def real_real_nn_baseline(real_df, feature_cols, class_value=None, n_sample=2000, seed=42):
#     if class_value is None:
#         X = real_df[feature_cols].values
#         tag = "ALL"
#     else:
#         X = real_df[real_df["Class"] == class_value][feature_cols].values
#         tag = f"Class={class_value}"

#     rng = np.random.default_rng(seed)
#     n = min(n_sample, len(X))
#     if n < 2:
#         return {"group": tag, "RealReal_NN_mean": np.nan, "RealReal_NN_min": np.nan, "n_used": n}

#     idx1 = rng.choice(len(X), n, replace=False)
#     idx2 = rng.choice(len(X), n, replace=False)
#     X1, X2 = X[idx1], X[idx2]

#     d = pairwise_distances(X1, X2, metric="euclidean")
#     # 자기 자신과의 거리(0)가 들어가면 최소가 0이 될 수 있으니 대각선 무시
#     np.fill_diagonal(d, np.inf)

#     min_d = d.min(axis=1)
#     return {"group": tag, "RealReal_NN_mean": float(min_d.mean()), "RealReal_NN_min": float(min_d.min()), "n_used": n}

# # 전체 baseline
# baseline_all = real_real_nn_baseline(train_base, FEATURE_COLS, class_value=None, n_sample=2000)

# # fraud baseline
# baseline_fraud = real_real_nn_baseline(train_base, FEATURE_COLS, class_value=1, n_sample=2000)

# baseline_all, baseline_fraud


In [24]:
# baseline_df = pd.DataFrame([
#     {"syn_ratio": "BASELINE_REALREAL_ALL",   "NN_mean": baseline_all["RealReal_NN_mean"],   "NN_min": baseline_all["RealReal_NN_min"]},
#     {"syn_ratio": "BASELINE_REALREAL_FRAUD", "Fraud_NN_mean": baseline_fraud["RealReal_NN_mean"], "Fraud_NN_min": baseline_fraud["RealReal_NN_min"]},
# ]).set_index("syn_ratio")

# display(privacy_df)
# display(baseline_df)


In [25]:
# imbalance_df = pd.DataFrame(imbalance_rows)

# # x축에 사용할 요청 ratio 값(자연비율은 0으로 표기하거나 NaN으로 둬도 됨)
# imbalance_df["requested_r"] = imbalance_df["syn_ratio_setting"].apply(
#     lambda s: 0.0 if s == "natural" else float(s.replace("forced_", ""))
# )

# imbalance_df = imbalance_df.sort_values("requested_r")

# plt.figure(figsize=(6,4), dpi=150)
# plt.plot(imbalance_df["requested_r"], imbalance_df["hybrid_fraud_ratio"], marker="o")
# plt.xlabel("Requested fraud ratio (r)  (natural shown as 0)")
# plt.ylabel("Actual fraud ratio in hybrid")
# plt.title("Imbalance mitigation via synthetic data")
# plt.grid(True, linestyle="dotted")
# plt.tight_layout()
# plt.show()

# imbalance_df

In [26]:
# from sklearn.manifold import TSNE
# from sklearn.preprocessing import StandardScaler
# import matplotlib.pyplot as plt

# def tsne_real_vs_syn(real_df, syn_df, label_col="Class",
#                      sample_per_group=5000, perplexity=30, random_state=42,
#                      point_size=10, alpha=0.75):
#     # --- 디자인 스타일 (회색 배경 + 흰색 그리드) ---
#     plt.style.use("seaborn-v0_8")  # seaborn 느낌

#     real_X = real_df.drop(columns=[label_col]).sample(
#         n=min(sample_per_group, len(real_df)), random_state=random_state
#     )
#     syn_X = syn_df.drop(columns=[label_col]).sample(
#         n=min(sample_per_group, len(syn_df)), random_state=random_state
#     )

#     X = pd.concat([real_X, syn_X], axis=0).reset_index(drop=True)
#     domain = np.array(["Real"]*len(real_X) + ["Synthetic"]*len(syn_X))

#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X)

#     Z = TSNE(
#         n_components=2,
#         perplexity=perplexity,
#         init="pca",
#         learning_rate="auto",
#         random_state=random_state
#     ).fit_transform(X_scaled)

#     # --- plot ---
#     fig, ax = plt.subplots(figsize=(7, 6), dpi=150)

#     # 배경색 + 흰색 그리드
#     ax.set_facecolor("#EAEAF2")
#     ax.grid(True, color="white", linewidth=1.2)
#     ax.set_axisbelow(True)

#     # 테두리(스파인) 제거 → 깔끔
#     for spine in ax.spines.values():
#         spine.set_visible(False)

#     # 점 그리기 (명칭은 그대로 Real / Synthetic)
#     for name in ["Real", "Synthetic"]:
#         idx = domain == name
#         ax.scatter(
#             Z[idx, 0], Z[idx, 1],
#             s=point_size, alpha=alpha,
#             edgecolors="none",  # 점 테두리 제거
#             label=name
#         )

#     # 축 라벨은 필요 없으면 감추기(원하면 주석 해제)
#     ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

#     # 범례 깔끔하게
#     ax.legend(frameon=True, framealpha=0.9)

#     plt.tight_layout()
#     plt.show()


# def tsne_fraud_only(real_df, syn_df, sample=2000):
#     tsne_real_vs_syn(
#         real_df[real_df.Class==1],
#         syn_df[syn_df.Class==1],
#         sample_per_group=sample,
#         perplexity=20
#     )

# # natural(None)도 포함
# ratios_ext = [None, 0.1, 0.5, 1.0]

# for r_vis in ratios_ext:
#     ratio_name = "natural" if r_vis is None else f"forced_{r_vis:.2f}"
#     print(f"t-SNE Diffusion ratio={ratio_name}")

#     syn_vis = sample_diffusion(
#         synthesizer_model, diffuser_model, diff_scaler,
#         n=len(train_base), ratio=r_vis, device=device
#     )

#     tsne_real_vs_syn(train_base, syn_vis)
#     tsne_fraud_only(train_base, syn_vis)

Visualize training loss progression.

In [27]:
# # prepare plot
# fig = plt.figure()
# ax = fig.add_subplot(111)

# # add grid
# ax.grid(linestyle='dotted')

# # plot the training epochs vs. the epochs' classification error
# ax.plot(np.array(range(1, len(train_epoch_losses)+1)), train_epoch_losses, label='epoch loss (blue)')

# # add axis legends
# ax.set_xlabel('[Training Epoch $e_i$]', fontsize=10)
# ax.set_ylabel('[MSE Error $\mathcal{L}^{MSE}$]', fontsize=10)

# # set plot legend
# plt.legend(loc='upper right', numpoints=1, fancybox=True)

# # add plot title
# plt.title('FinDiff Training Epochs $e_i$ vs. MSE Error $L^{MSE}$', fontsize=10);

In [28]:
# from scipy.stats import ks_2samp, wasserstein_distance

# def distribution_similarity(real_df, syn_df, exclude_cols=["Class"]):
#     ks_scores, wass_scores = [], []
#     for col in real_df.columns:
#         if col not in exclude_cols:
#             ks_scores.append(ks_2samp(real_df[col], syn_df[col]).statistic)
#             wass_scores.append(wasserstein_distance(real_df[col], syn_df[col]))
#     return {"KS_mean": np.mean(ks_scores),
#             "Wasserstein_mean": np.mean(wass_scores)}

# def fraud_collapse_check(syn_df, threshold=0.001):
#     fr = syn_df.Class.mean()
#     return {"Fraud_Ratio": fr, "Collapsed": fr < threshold}

# ratios_ext = [None] + ratios  # natural + forced들

# quality_results = {}

# for r in ratios_ext:
#     ratio_name = "natural" if r is None else f"forced_{r:.2f}"

#     syn = sample_diffusion(
#         synthesizer_model, diffuser_model, diff_scaler,
#         n=len(train_base), ratio=r, device=device
#     )

#     real_sample = train_base.sample(
#         n=min(len(train_base), len(syn)),
#         replace=False,
#         random_state=42
#     )
#     syn_sample = syn.sample(n=len(real_sample), random_state=42)

#     quality_results[ratio_name] = {
#         **distribution_similarity(real_sample, syn_sample),
#         **fraud_collapse_check(syn),
#         "Diffusion_Loss_Var": np.var(train_epoch_losses[-100:])
#     }

# quality_df = pd.DataFrame(quality_results).T
# quality_df.to_csv("diffusion_quality_with_natural.csv")
# quality_df

# **Generate Data using the FinDiff model**


Double-click (or enter) to edit

Init and set sampling parameters.

Use FinDiff to generate new data samples.

Decode generated FinDiff samples.

In [29]:
# syn_example = sample_diffusion(
#     synthesizer_model, diffuser_model, diff_scaler, 10_000, ratio=0.5
# )
# print("Fraud ratio:", syn_example["Class"].mean())
# syn_example.head()
